In [1]:
import os, shutil
from google.colab import drive

# 1. 挂载 Drive
drive.mount('/content/drive')

# 2. 路径设置
DRIVE_DS = "/content/drive/MyDrive/COMP4330_Whisper_MCE/mce_hf_dataset"
LOCAL_DS = "/content/mce_hf_dataset"

# 3. 清理 Drive 上可能被中断产生的脏缓存文件 (防止复制过来报错)
for split in ["train", "test"]:
    d = os.path.join(DRIVE_DS, split)
    if os.path.isdir(d):
        for f in os.listdir(d):
            if f.startswith("cache-") and f.endswith(".arrow"):
                os.remove(os.path.join(d, f))

# 4. 复制数据集到本地 /content
if not os.path.exists(LOCAL_DS):
    print("正在从 Drive 复制数据集到本地（约 3-5 分钟）...")
    shutil.copytree(DRIVE_DS, LOCAL_DS)
    print("✅ 复制完成")
else:
    print("✅ 本地已存在数据集，跳过复制")

# 检查空间
!df -h /content | tail -1

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ 本地已存在数据集，跳过复制
overlay         236G  7.8G  228G   4% /


In [2]:
!pip install -U bitsandbytes>=0.46.1

In [3]:
import torch
from transformers import (
    WhisperProcessor, WhisperForConditionalGeneration,
    BitsAndBytesConfig, Seq2SeqTrainingArguments, Seq2SeqTrainer,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from datasets import load_from_disk
from dataclasses import dataclass
from typing import Any, Dict, List, Union
import numpy as np

# ====================== 0. 从本地加载数据 ======================
dataset = load_from_disk("/content/mce_hf_dataset")
print(dataset)

# ====================== 1. 4-bit 量化设定 ======================
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model_name = "openai/whisper-small"
processor = WhisperProcessor.from_pretrained(model_name, language="zh", task="transcribe")

model = WhisperForConditionalGeneration.from_pretrained(
    model_name, quantization_config=bnb_config, device_map="auto",
)
model.config.forced_decoder_ids = None
model.config.suppress_tokens = []
model.generation_config.language = "zh"
model.generation_config.task = "transcribe"
model = prepare_model_for_kbit_training(model)

# ====================== 2. LoRA 设定 ======================
lora_config = LoraConfig(
    r=16, lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05, bias="none",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# ====================== 3. 数据预处理 ======================
def prepare_dataset(batch):
    audio = batch["audio"]
    if hasattr(audio, "get_all_samples"):
        s = audio.get_all_samples()
        array, sr = s.data.numpy().squeeze(), s.sample_rate
    else:
        array, sr = audio["array"], audio["sampling_rate"]

    if array.ndim > 1:
        array = array.mean(axis=0)
    array = array.astype(np.float32)

    batch["input_features"] = processor(array, sampling_rate=sr).input_features[0]
    batch["labels"] = processor.tokenizer(batch["sentence"]).input_ids
    return batch

print("處理訓練資料中...")
train_dataset = dataset["train"].map(
    prepare_dataset,
    remove_columns=dataset["train"].column_names,
    desc="Processing train",
)
print("處理完成：", train_dataset)

# ====================== 4. Data Collator ======================
@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any
    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        input_features = [{"input_features": f["input_features"]} for f in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")
        label_features = [{"input_ids": f["labels"]} for f in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)
        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]
        batch["labels"] = labels
        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)

# ====================== 5. 训练参数 ======================
training_args = Seq2SeqTrainingArguments(
    output_dir="/content/whisper-small-mce-qlora",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=1e-4,
    warmup_steps=100,
    max_steps=1500,
    fp16=True,
    eval_strategy="no",
    save_steps=500,
    logging_steps=50,
    report_to=["none"],
    predict_with_generate=True,
    generation_max_length=225,
    save_total_limit=2,
    remove_unused_columns=False,
    label_names=["labels"],
    optim="paged_adamw_8bit",
    dataloader_num_workers=0,
)

# ====================== 6. 开始训练 ======================
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=data_collator,
    processing_class=processor.feature_extractor,
)

print("🚀 開始 QLoRA Fine-tuning...")
trainer.train()

# 只保存最终权重在本地
trainer.save_model("/content/whisper-small-mce-qlora-final")
processor.save_pretrained("/content/whisper-small-mce-qlora-final")
print("✅ 訓練完成，暫存在本地：/content/whisper-small-mce-qlora-final")

DatasetDict({
    train: Dataset({
        features: ['id', 'audio', 'sentence', 'topic', 'duration'],
        num_rows: 12643
    })
    test: Dataset({
        features: ['id', 'audio', 'sentence', 'topic', 'duration'],
        num_rows: 1405
    })
})


model.safetensors: reconstructing file:   0%|          |  0.00B /  967MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/3.87k [00:00<?, ?B/s]

trainable params: 1,769,472 || all params: 243,504,384 || trainable%: 0.7267
處理訓練資料中...


... (2 outputs left out)

/usr/local/lib/python3.13/dist-packages/torch/_dynamo/eval_frame.py:1548: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
50,10.523584
100,5.437179
150,3.259376
200,2.810715
250,2.484113
300,2.239065
350,2.212648
400,2.162053
450,2.092440
500,2.004703


/usr/local/lib/python3.13/dist-packages/torch/_dynamo/eval_frame.py:1548: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.13/dist-packages/torch/_dynamo/eval_frame.py:1548: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


✅ 訓練完成，暫存在本地：/content/whisper-small-mce-qlora-final


In [4]:
import os, shutil, json

SRC_LOCAL = "/content/whisper-small-mce-qlora-final"
DST_DRIVE = "/content/drive/MyDrive/COMP4330_Whisper_MCE/whisper-small-mce-qlora-final"

# 1. 保存模型和 Processor 到 Drive
if os.path.exists(DST_DRIVE):
    print("检测到 Drive 已有同名文件夹，正在覆盖...")
    shutil.rmtree(DST_DRIVE)

shutil.copytree(SRC_LOCAL, DST_DRIVE)
print(f"✅ QLoRA 权重和配置已成功保存到 Drive：\n{DST_DRIVE}")

# 2. 保存训练日志（可选，方便你复盘）
LOG_SRC = "/content/whisper-small-mce-qlora/trainer_state.json"
if os.path.exists(LOG_SRC):
    LOG_DST = "/content/drive/MyDrive/COMP4330_Whisper_MCE/qlora_trainer_state.json"
    with open(LOG_SRC) as f:
        state = json.load(f)
    with open(LOG_DST, "w") as f:
        json.dump(state, f, indent=2)
    print(f"✅ 训练日志已保存到：\n{LOG_DST}")

# 3. 验证 Drive 里的文件清单
print("\n=== Drive 最终保存的文件清单 ===")
!ls -lh {DST_DRIVE}

✅ QLoRA 权重和配置已成功保存到 Drive：
/content/drive/MyDrive/COMP4330_Whisper_MCE/whisper-small-mce-qlora-final

=== Drive 最终保存的文件清单 ===
total 11M
-rw------- 1 root root 1.2K Nov  8 02:17 adapter_config.json
-rw------- 1 root root 6.8M Nov  8 02:17 adapter_model.safetensors
-rw------- 1 root root  315 Nov  8 02:17 preprocessor_config.json
-rw------- 1 root root  409 Nov  8 02:17 processor_config.json
-rw------- 1 root root 5.1K Nov  8 02:17 README.md
-rw------- 1 root root 2.1K Nov  8 02:17 tokenizer_config.json
-rw------- 1 root root 3.8M Nov  8 02:17 tokenizer.json
-rw------- 1 root root 5.3K Nov  8 02:17 training_args.bin
